- Perform sentiment classification (positive vs negative reviews)



In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

data_path = Path('data/amazon_reviews_preprocessed.csv')
if not data_path.exists():
    raise FileNotFoundError(f"Preprocessed dataset not found: {data_path}")

df = pd.read_csv(data_path)
if not {'Star Rating', 'review_text_final'}.issubset(df.columns):
    raise KeyError("Expected 'Star Rating' and 'review_text_final' columns in the dataset.")

df['rating_num'] = df['Star Rating'].str.extract(r'(\d+(?:\.\d+)?)').astype(float)

# Keep clear binary labels for the mini use case.
sentiment_df = df[df['rating_num'] != 3].copy()
sentiment_df['sentiment'] = sentiment_df['rating_num'].apply(
    lambda rating: 'positive' if rating >= 4 else 'negative'
)

X = sentiment_df['review_text_final'].fillna('').astype(str)
y = sentiment_df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

class_distribution_df = y.value_counts().rename_axis('sentiment').reset_index(name='count')

print(f"Total reviews used for classification: {len(sentiment_df)}")
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

class_distribution_df


Total reviews used for classification: 624
Training set size: 499
Test set size: 125


,sentiment,count
0,positive,555
1,negative,69


- Use Logistic Regression or Naive Bayes

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

# Use the same classifier for both feature types so the comparison reflects the features.
classifier = LogisticRegression(
    max_iter=2000,
    random_state=42,
    class_weight='balanced',
)

def train_and_evaluate(vectorizer, feature_name):
    X_train_vectorized = vectorizer.fit_transform(X_train)
    X_test_vectorized = vectorizer.transform(X_test)

    model = LogisticRegression(
        max_iter=2000,
        random_state=42,
        class_weight='balanced',
    )
    model.fit(X_train_vectorized, y_train)

    y_pred = model.predict(X_test_vectorized)

    metrics = {
        'Feature Type': feature_name,
        'Train Shape': str(X_train_vectorized.shape),
        'Test Shape': str(X_test_vectorized.shape),
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision (positive)': round(
            precision_score(y_test, y_pred, pos_label='positive', zero_division=0), 4
        ),
        'Recall (positive)': round(
            recall_score(y_test, y_pred, pos_label='positive', zero_division=0), 4
        ),
        'F1 Score (positive)': round(
            f1_score(y_test, y_pred, pos_label='positive', zero_division=0), 4
        ),
        'Macro F1': round(
            f1_score(y_test, y_pred, average='macro', zero_division=0), 4
        ),
    }

    report_df = pd.DataFrame(
        classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    ).transpose().round(4)

    return {
        'model': model,
        'vectorizer': vectorizer,
        'metrics': metrics,
        'report_df': report_df,
    }

print('Classifier selected: Logistic Regression')
print('Reason: it provides a strong linear baseline and allows a fair BoW vs TF-IDF comparison using the same model.')


Classifier selected: Logistic Regression
Reason: it provides a strong linear baseline and allows a fair BoW vs TF-IDF comparison using the same model.


- Compare performance using BoW and TF-IDF features

In [3]:
from IPython.display import display

bow_results = train_and_evaluate(
    CountVectorizer(token_pattern=r'(?u)\b\w+\b'),
    'Bag of Words',
)

tfidf_results = train_and_evaluate(
    TfidfVectorizer(token_pattern=r'(?u)\b\w+\b'),
    'TF-IDF',
)

performance_comparison_df = pd.DataFrame(
    [bow_results['metrics'], tfidf_results['metrics']]
).sort_values(by=['Macro F1', 'F1 Score (positive)', 'Accuracy'], ascending=False).reset_index(drop=True)

best_feature = performance_comparison_df.iloc[0]['Feature Type']

print('Performance Comparison:')
display(performance_comparison_df)

print('Classification Report for Bag of Words:')
display(bow_results['report_df'])

print('Classification Report for TF-IDF:')
display(tfidf_results['report_df'])

print(f"Best performing feature set on this split: {best_feature}")


Performance Comparison:


,Feature Type,Train Shape,Test Shape,Accuracy,Precision (positive),Recall (positive),F1 Score (positive),Macro F1
0,TF-IDF,"(499, 2931)","(125, 2931)",0.976,0.9909,0.982,0.9864,0.9415
1,Bag of Words,"(499, 2931)","(125, 2931)",0.968,0.9908,0.973,0.9818,0.9242


Classification Report for Bag of Words:


,precision,recall,f1-score,support
negative,0.8125,0.9286,0.8667,14.000
positive,0.9908,0.9730,0.9818,111.000
accuracy,0.9680,0.9680,0.9680,0.968
macro avg,0.9017,0.9508,0.9242,125.000
weighted avg,0.9709,0.9680,0.9689,125.000


Classification Report for TF-IDF:


,precision,recall,f1-score,support
negative,0.8667,0.9286,0.8966,14.000
positive,0.9909,0.9820,0.9864,111.000
accuracy,0.9760,0.9760,0.9760,0.976
macro avg,0.9288,0.9553,0.9415,125.000
weighted avg,0.9770,0.9760,0.9764,125.000


Best performing feature set on this split: TF-IDF
